# Silver Layer Tutorial: Bronze → Cleaned Data

This notebook walks through transforming **bronze** IMDb reviews into the **silver** layer by applying text cleaning.

**Data flow:**

1. Read JSONL from MinIO `bronze/imdb/`
2. Apply `clean_text()` (lowercase, HTML strip, whitespace normalization)
3. Deduplicate on `(text, label)` to avoid redundant records for embedding/training
4. Write cleaned records to `silver/imdb/`

**Prerequisites:**

- Bronze data already in MinIO (run `stream_to_bronze_tutorial.ipynb` in this folder or the producer + bronze consumer first)
- MinIO running: `docker compose -f docker/docker-compose.yml up -d minio`

In [ ]:
!pip install -q boto3

## 1. Schema: `ImdbSilverReview`

The silver layer uses **`ImdbSilverReview`** (same shape as bronze but with cleaned text):

| Field  | Type | Description |
|--------|------|-------------|
| `id`   | str  | Unique identifier |
| `text` | str  | **Cleaned** review text |
| `label`| 0 or 1 | Binary sentiment |

## 2. Text cleaning

`src.transformation.cleaning` provides:
- **`strip_html(text)`** – remove HTML tags
- **`normalize_whitespace(text)`** – collapse and trim whitespace
- **`clean_text(text)`** – full pipeline: HTML strip → whitespace → lowercase

In [ ]:
from src.transformation.cleaning import clean_text

raw = "  <p>GREAT   MOVIE!</p>  "
cleaned = clean_text(raw)
print(f"Before: {repr(raw)}")
print(f"After:  {repr(cleaned)}")

## 2b. Deduplication

`deduplicate_by_text_label()` keeps the first occurrence of each `(text, label)` pair. Duplicates (same cleaned text + same label) are dropped to avoid redundant BERT embeddings and training bias.

In [ ]:
from src.transformation.silver_job import deduplicate_by_text_label
from src.utils.schema import ImdbSilverReview

# Same text+label duplicated
recs = [
    ImdbSilverReview(id="1", text="great movie", label=1),
    ImdbSilverReview(id="2", text="great movie", label=1),  # duplicate
    ImdbSilverReview(id="3", text="bad film", label=0),
]
deduped = deduplicate_by_text_label(recs)
print(f"Before: {len(recs)} records, after: {len(deduped)} (duplicate removed)")

## 3. Read bronze from MinIO

List and read one bronze JSONL file.

In [ ]:
import json
from src import config
from src.utils.s3_client import get_s3_client, list_objects, get_object_body

prefix = f"{config.BRONZE_PREFIX}imdb/"
objects = list_objects(prefix)
print(f"Bronze objects under {prefix}: {len(objects)}")
for obj in objects[:3]:
    print(f"  - {obj['Key']}")

if not objects:
    raise SystemExit("No bronze data. Run stream_to_bronze_tutorial first.")

# Pick first JSONL file
first_key = next(o["Key"] for o in objects if o["Key"].endswith(".jsonl"))
body = get_object_body(first_key)
lines = [l.strip() for l in body.decode("utf-8").strip().split("\n") if l.strip()]
print(f"\nSample bronze record (first line):")
print(json.dumps(json.loads(lines[0]), indent=2)[:500])

## 4. Apply cleaning, deduplicate, and build silver records

Transform each bronze record into a silver record. Deduplication keeps the first occurrence of each `(text, label)` pair to avoid redundant records for downstream BERT embedding and training.

In [ ]:
from src.transformation.cleaning import clean_text
from src.transformation.silver_job import deduplicate_by_text_label
from src.utils.schema import ImdbSilverReview

silver_records = []
for line in lines:
    rec = json.loads(line)
    raw_text = str(rec.get("text", ""))
    cleaned = clean_text(raw_text)
    silver = ImdbSilverReview(
        id=str(rec.get("id")),
        text=cleaned,
        label=int(rec.get("label", 0)),
    )
    silver_records.append(silver)

# Deduplicate on (text, label) - keeps first occurrence
silver_records = deduplicate_by_text_label(silver_records)
print(f"Processed {len(silver_records)} records (after deduplication)")
if silver_records:
    print(f"Example before: {json.loads(lines[0])['text'][:80]}...")
    print(f"Example after:  {silver_records[0].text[:80]}...")

## 5. Write silver to MinIO

Upload cleaned records as JSONL to the silver layer.

In [ ]:
import time
from uuid import uuid4
from src.utils.s3_client import ensure_bucket_exists, upload_bytes

ensure_bucket_exists()
silver_prefix = f"{config.SILVER_PREFIX}imdb/"
key = f"{silver_prefix}imdb_silver_{int(time.time())}_{uuid4().hex}.jsonl"
data = ("\n".join(json.dumps(s.to_dict(), ensure_ascii=False) for s in silver_records) + "\n").encode("utf-8")
upload_bytes(key, data)
print(f"Wrote {len(silver_records)} records to s3://{config.S3_DATA_BUCKET}/{key}")

## 6. Verify silver objects

List silver files and show a sample.

In [ ]:
silver_prefix = f"{config.SILVER_PREFIX}imdb/"
silver_objects = list_objects(silver_prefix)
print(f"Silver objects: {len(silver_objects)}")
for obj in silver_objects[:5]:
    print(f"  - {obj['Key']}")

if silver_objects:
    skey = next(o["Key"] for o in silver_objects if o["Key"].endswith(".jsonl"))
    sbody = get_object_body(skey)
    slines = [l for l in sbody.decode("utf-8").strip().split("\n") if l]
    print(f"\nFirst silver record from {skey}:")
    print(json.dumps(json.loads(slines[0]), indent=2))

## 7. Run via CLI

To process all bronze files at once (with deduplication enabled by default):

```bash
python -m src.transformation.silver_job
```

To disable deduplication:

```bash
python -m src.transformation.silver_job --no-dedup
```